## EXP-INGEST-002 — Content & Metadata Extraction

### Purpose

After inspecting the uploaded file, I want CEREBRO to extract as much useful information as possible without AI.

This establishes which artifact-form fields can be populated directly from the source before AI is used to suggest missing or richer metadata.

### Extract the data

In [16]:
from pathlib import Path
import hashlib
import mimetypes

# Resolve repository and source artifact
repo_root = Path.cwd().parents[1]

source_path = (
    repo_root
    / "poc/data/raw/text/benchmark_001.txt"
)

assert source_path.exists(), (
    f"Source artifact not found: {source_path}"
)

# Read source
file_bytes = source_path.read_bytes()

# Determine file facts required by 02B
mime_type, _ = mimetypes.guess_type(source_path.name)

file_facts = {
    "filename": source_path.name,
    "extension": source_path.suffix.lower(),
    "mime_type": mime_type,
    "size_bytes": len(file_bytes),
    "sha256": hashlib.sha256(file_bytes).hexdigest(),
    "encoding": "utf-8"
}

# Extract text
source_text = file_bytes.decode(
    file_facts["encoding"]
)

print("✓ Source artifact loaded")
print("File       :", file_facts["filename"])
print("MIME       :", file_facts["mime_type"])
print("Encoding   :", file_facts["encoding"])
print("Characters :", len(source_text))
print("Words      :", len(source_text.split()))

print("\n--- Extracted Content ---")
print(source_text)

✓ Source artifact loaded
File       : benchmark_001.txt
MIME       : text/plain
Encoding   : utf-8
Characters : 294
Words      : 41

--- Extracted Content ---
CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge. It maintains provenance between knowledge fragments and their original source artifacts. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### Basic deterministic content statistics

In [17]:
lines = source_text.splitlines()
words = source_text.split()

content_facts = {
    "character_count": len(source_text),
    "word_count": len(words),
    "line_count": len(lines),
    "non_empty_line_count": len(
        [line for line in lines if line.strip()]
    )
}

content_facts

{'character_count': 294,
 'word_count': 41,
 'line_count': 1,
 'non_empty_line_count': 1}

### Extract deterministic metadata

In [18]:
extracted_metadata = {
    "title": None,
    "author": None,
    "created_date": None,
    "language": None,
    "description": None,
    "topics": [],
    "people": [],
    "organizations": [],
    "projects": [],
    "tags": []
}

# Filename-derived title candidate
extracted_metadata["title"] = source_path.stem

extracted_metadata

{'title': 'benchmark_001',
 'author': None,
 'created_date': None,
 'language': None,
 'description': None,
 'topics': [],
 'people': [],
 'organizations': [],
 'projects': [],
 'tags': []}

In [19]:
{
    "title": "benchmark_001",
    "author": None,
    "created_date": None,
    "language": None,
    "description": None,
    "topics": [],
    "people": [],
    "organizations": [],
    "projects": [],
    "tags": []
}

{'title': 'benchmark_001',
 'author': None,
 'created_date': None,
 'language': None,
 'description': None,
 'topics': [],
 'people': [],
 'organizations': [],
 'projects': [],
 'tags': []}

### Add field provenance

In [20]:
prefill_fields = {
    "title": {
        "value": source_path.stem,
        "source": "filename",
        "method": "deterministic",
        "user_confirmed": False
    },

    "author": {
        "value": None,
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "language": {
        "value": None,
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "description": {
        "value": None,
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "topics": {
        "value": [],
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "people": {
        "value": [],
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "organizations": {
        "value": [],
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "projects": {
        "value": [],
        "source": None,
        "method": None,
        "user_confirmed": False
    },

    "tags": {
        "value": [],
        "source": None,
        "method": None,
        "user_confirmed": False
    }
}

### Build the prefill record

In [21]:
artifact_prefill = {
    "status": "staged",

    "file_facts": file_facts,

    "content_facts": content_facts,

    "fields": prefill_fields,

    "processing": {
        "file_inspection": True,
        "content_extraction": True,
        "ai_enrichment": False,
        "user_reviewed": False,
        "submitted": False
    }
}

artifact_prefill

{'status': 'staged',
 'file_facts': {'filename': 'benchmark_001.txt',
  'extension': '.txt',
  'mime_type': 'text/plain',
  'size_bytes': 294,
  'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832',
  'encoding': 'utf-8'},
 'content_facts': {'character_count': 294,
  'word_count': 41,
  'line_count': 1,
  'non_empty_line_count': 1},
 'fields': {'title': {'value': 'benchmark_001',
   'source': 'filename',
   'method': 'deterministic',
   'user_confirmed': False},
  'author': {'value': None,
   'source': None,
   'method': None,
   'user_confirmed': False},
  'language': {'value': None,
   'source': None,
   'method': None,
   'user_confirmed': False},
  'description': {'value': None,
   'source': None,
   'method': None,
   'user_confirmed': False},
  'topics': {'value': [],
   'source': None,
   'method': None,
   'user_confirmed': False},
  'people': {'value': [],
   'source': None,
   'method': None,
   'user_confirmed': False},
  'organizations': {'value': []

### Identify missing metadata

In [22]:
missing_fields = []

for field, metadata in prefill_fields.items():

    value = metadata["value"]

    if value is None or value == []:
        missing_fields.append(field)

print("Fields populated deterministically:")
for field in prefill_fields:
    if field not in missing_fields:
        print("✓", field)

print("\nFields requiring enrichment:")
for field in missing_fields:
    print("○", field)

Fields populated deterministically:
✓ title

Fields requiring enrichment:
○ author
○ language
○ description
○ topics
○ people
○ organizations
○ projects
○ tags


### Intake state

In [23]:
print("CEREBRO Artifact Intake")
print("-----------------------")

print("File       :", file_facts["filename"])
print("MIME       :", file_facts["mime_type"])
print("Encoding   :", file_facts["encoding"])
print("Words      :", content_facts["word_count"])

print(
    "Title      :",
    prefill_fields["title"]["value"],
    "(from filename)"
)

print("\nMissing metadata:", len(missing_fields))

print("\nStatus      : STAGED")
print("AI enrichment: PENDING")
print("User review  : PENDING")

CEREBRO Artifact Intake
-----------------------
File       : benchmark_001.txt
MIME       : text/plain
Encoding   : utf-8
Words      : 41
Title      : benchmark_001 (from filename)

Missing metadata: 8

Status      : STAGED
AI enrichment: PENDING
User review  : PENDING


### Expected conceptual state

flowchart LR
    U[Upload] --> FI[File Inspection]
    FI --> CE[Content Extraction]
    CE --> DM[Deterministic Metadata]
    DM --> GAP[Identify Metadata Gaps]
    GAP --> AI[AI Enrichment]

### Result

CEREBRO successfully extracted the artifact's readable content and deterministic metadata without using AI.

The system also identified which metadata fields remain unresolved.

Instead of sending the entire intake process to an AI model, CEREBRO can now use AI selectively to enrich only the information that cannot be reliably obtained from the artifact itself.